## LORA training/testing pipeline — Task 1 (Risk Clause Recognition), Hard Negatives

This notebook implements the fine-tuning pipeline for **Task 1: binary clause identification**, following [TASK1_HARD_NEGATIVES_PLAN.md](docs/TASK1_HARD_NEGATIVES_PLAN.md) (Option A — Hard Negatives).

Task 1 = given a single contract clause excerpt, decide whether it is an instance of a specific risk clause category (`Yes`) or not (`No`) across the 32 Yes/No categories.

**The fix this notebook applies (closing the leak):** in the raw CSV, a category's input text is non-empty *exactly when* the answer is `Yes`. Training naively on that lets the model cheat — "any real text → Yes, placeholder → No" — without ever reading a clause. Instead, every `No` example **borrows a real clause from a different category in the same contract** (a *hard negative*), so both `Yes` and `No` inputs are genuine legal text and the model must actually recognize the clause type.

**Note:** this notebook loads the *sampled* CSV (`master_clauses_cleaned_sampled.csv`, ~100 records) for quick iteration instead of the full dataset.

In [ ]:
import sys; print("UTF-8 mode:", sys.flags.utf8_mode)

# Step 1 : Load data (sampled master_clauses file from CUAD)

Dataset Description Summarized : 

1. Columns NOT ending in "Answer" (Context Columns)
- Role: These columns contain the text context (the actual excerpt or "clause") extracted from the contract.
- Content: A string of text directly from the contract that is responsive to a specific category.
- Purpose: This serves as the "evidence" or the "source passage" that justifies a specific determination.
- Handling of Omissions: If parts of the text are irrelevant, they may be replaced with <omitted>.

2. Columns ending in "Answer" (Label Columns)
- Role: These columns contain the derived human-input answers based on the text context found in the corresponding Context column.
- Content:
- For "Yes/No" Categories (32 types): The value is "Yes" if the clause exists, or "No" if no string was found. (e.g., Termination for Convenience).
- For Extraction Categories (Task 2): The value is a normalized string representing a specific entity, date, or number.
- Purpose: This is the "ground truth" or "label" for the machine learning task.

In [ ]:
import pandas as pd
import json
from pathlib import Path
import csv
import re
from sklearn.model_selection import train_test_split

In [ ]:
CUAD_PATH = Path('data/CUAD_v1')
# Step 1: load the cleaned CSV.
# The plan calls for the full master_clauses_cleaned.csv (510 contracts); for quick
# iteration we use the sampled file (~100 records) — the only deviation from the plan.
# MASTER_CLAUSES_PATH = CUAD_PATH/'master_clauses_cleaned.csv'        # full dataset
MASTER_CLAUSES_PATH = CUAD_PATH/'master_clauses_cleaned_sampled.csv'  # ~100 records, quick iteration

try:
    # Read the file manually using the CSV module first to handle inconsistencies
    data = []
    with open(MASTER_CLAUSES_PATH, 'r', encoding='utf-8', errors='replace') as f:
        # Use csv.Sniffer to deduce format if possible, or enforce standard strictness
        reader = csv.DictReader(f) 
        for i, row in enumerate(reader):
            data.append(row)

    # Convert the list of dicts to a DataFrame
    df = pd.DataFrame(data)

    print(f"Data Loaded Successfully via CSV module.")
    print(f"Total Contracts: {len(df)}")
    print(df.head(3))

except Exception as e:
    print(f"Error: {e}")

In [ ]:
df.head(3)

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    # Remove special characters but keep spaces
    return re.sub(r'[^a-zA-Z0-9\s]', '', text)

# Clean column names
df.columns = [clean_text(col).strip() for col in df.columns]
for col in df.columns:
    print(col)

In [ ]:
new_columns = {}
for col in df.columns:
    print(f"Processing column: '{col}'")
    if "Answer" in col:
            # Remove "Answer" from the string and append "_Answer" at the end
            new_columns[col] = f"{col.replace('Answer', '').strip()}_Answer"

df = df.rename(columns=new_columns)

In [ ]:
for col in df.columns:
    print(col)

# Step 2 (cont.) : Restrict to the 32 Task 1 categories

Per the plan, the load / column-clean / `_Answer`-rename cells above are left unchanged. Here we derive the Task 1 category list by excluding the Task 2 entity-extraction fields, so Task 2 fields never enter Task 1 training.

In [ ]:
task2_categories = [
    "Filename", "Document Name", "Parties", "Agreement Date", "Effective Date",
    "Expiration Date", "Renewal Term", "Notice Period To Terminate Renewal",
    "Governing Law", "Warranty Duration",
]
task1_categories = [
    col for col in df.columns
    if not col.endswith("_Answer") and col.strip() != ""
    and col not in task2_categories
    and f"{col}_Answer" in df.columns
]
assert len(task1_categories) == 32, f"Expected 32 Task 1 categories, got {len(task1_categories)}"
print(f"{len(task1_categories)} Task 1 categories:")
for c in task1_categories:
    print(" -", c)

In [ ]:
def save_jsonl(data, filename):
    with open(filename, 'w') as f:
        for entry in data:
            f.write(json.dumps(entry) + '\n')

# Step 3 & 4 : Build binary (Yes/No) examples with **hard negatives**, split by contract

For each contract row we gather **all clauses actually present** in that contract, keyed by category. Then for each of the 32 categories we build one example:

- **Yes** → use that category's own real clause text.
- **No** → randomly borrow a real clause from a **different** present category (a *hard negative*). If the contract has no other clause to borrow, skip the example.

So both `Yes` and `No` inputs are genuine legal text — the model can no longer cheat off a placeholder, and the only way to answer is to recognize the clause type. The instruction wording is *"Is the following contract text a `...` clause?"* because we now feed a single clause excerpt.

The split is done on **contracts (df rows) first** (Step 4), then examples are built from each side, to prevent a contract leaking across train/val.

In [ ]:
import random
random.seed(42)  # fix randomness so the same hard negatives are picked every run

# Turn a raw answer cell into a clean Yes/No label.
# "No" or blank -> "No"; anything else (real clause text) -> "Yes".
def to_binary(answer):
    return "No" if (pd.isna(answer) or str(answer).strip().lower() == "no") else "Yes"

# Return the clause text in a column, or None if that cell is empty/blank.
def nonempty_context(row, cat):
    v = row.get(cat)
    return str(v).strip() if pd.notna(v) and str(v).strip() else None

# GOAL OF THIS FUNCTION:
# Turn a table of contracts into individual training examples for Task 1.
# Each example is one yes/no question: "is THIS piece of text an example of
# category X?". The key trick (hard negatives) is that the "No" examples are NOT
# a placeholder — they are real clause text taken from a DIFFERENT category in the
# same contract, so the model has to actually understand the clause to answer.
def build_examples(frame):
    rows = []  # all finished examples, one dict per question
    for _, row in frame.iterrows():  # go through one contract at a time
        # Step 1: collect every clause that is actually written in THIS contract,
        # as {category name: clause text}. These real texts are the only ones we
        # are allowed to borrow from when we need a "No" example for this contract.
        present = {c: nonempty_context(row, c)
                   for c in task1_categories if nonempty_context(row, c)}
        # Step 2: ask the yes/no question once for every one of the 32 categories.
        for category in task1_categories:
            if to_binary(row[f"{category}_Answer"]) == "Yes":
                # Step 3a (YES case): the contract really has this clause, so use
                # the category's own real text as the input and label it "Yes".
                text, label = present[category], "Yes"
            else:
                # Step 3b (NO case = hard negative): the contract does NOT have
                # this clause. Instead of a placeholder, gather the real clauses
                # from all the OTHER categories present in this contract...
                others = [t for c, t in present.items() if c != category]
                if not others:
                    continue  # ...if there is nothing else to borrow, skip this one
                # ...and pick one of those real clauses at random, labelled "No".
                text, label = random.choice(others), "No"
            # Step 4: save the finished example: the question, the text, the answer.
            rows.append({
                "instruction": f'Is the following contract text a "{category}" clause? Answer strictly "Yes" or "No".',
                "category": category,
                "input": text,
                "output": label,
            })
    return rows

# Step 4 (pipeline): split by CONTRACT first, then build examples from each side,
# so no single contract's clauses end up in both train and validation.
train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)
train_data = build_examples(train_df)
val_data = build_examples(val_df)

print(f"Contracts — train: {len(train_df)}, val: {len(val_df)}")
print(f"Examples  — train: {len(train_data)}, val: {len(val_data)}")

In [ ]:
for train, val in zip(train_data[:3], val_data[:3]):
    print("TRAIN EXAMPLE:")
    print(json.dumps(train, indent=2))
    print("\nVAL EXAMPLE:")
    print(json.dumps(val, indent=2))
    print("\n" + "="*50 + "\n")

# Step 5 : Balance classes on the **train** split only

Hard negatives can still be a minority/majority depending on how many clauses each contract has, and without balancing the model drifts to always answering the majority class. Downsample the majority class toward ~1:1 on **train only**; leave `val_data` at its natural distribution so validation metrics stay honest.

In [ ]:
from collections import defaultdict

def balance(data, ratio=1.0, seed=42):
    rng = random.Random(seed)
    pos = [e for e in data if e["output"] == "Yes"]
    neg = [e for e in data if e["output"] == "No"]
    keep_neg = min(len(neg), int(len(pos) * ratio))
    neg = rng.sample(neg, keep_neg)
    out = pos + neg
    rng.shuffle(out)
    return out

train_data = balance(train_data, ratio=1.0)   # train only — val untouched
print(f"After balancing — train: {len(train_data)} examples")

# Step 6a : Sanity checks — class counts + verify hard negatives are real text

Two quick checks before saving:
1. Print the final `Yes`/`No` counts (train is balanced; val is natural).
2. **Inspect a few `No` examples** — their `input` must be *real clause text borrowed from another category*, never the old `[No matching clause excerpt found...]` placeholder. If a `No` input is a placeholder, the leak isn't closed.

The headline per-class precision / recall / F1 + confusion matrix is computed **after training** in Step 6b.

In [ ]:
from collections import Counter

def label_counts(data):
    return Counter(ex["output"] for ex in data)

print("Train label counts:", dict(label_counts(train_data)))
print("Val   label counts:", dict(label_counts(val_data)))

# Verify hard negatives: every `No` input must be REAL clause text, not a placeholder.
print("\nSample of `No` examples (inputs must be real borrowed clause text):")
no_examples = [ex for ex in train_data if ex["output"] == "No"]
for ex in no_examples[:3]:
    print(f"\n  category : {ex['category']}")
    print(f"  input    : {ex['input'][:200]}{'...' if len(ex['input']) > 200 else ''}")

assert all("[No matching clause excerpt found" not in ex["input"] for ex in no_examples), \
    "Found placeholder text in a No example — the leak is NOT closed."
print("\nOK — no placeholder strings found in `No` inputs.")

# Save examples to JSONL (ensure dirs exist; save paths == load paths)

In [ ]:
CUAD_TRAIN_PATH = CUAD_PATH/'train'
CUAD_VALIDATION_PATH = CUAD_PATH/'validation'

# Ensure the train/ and validation/ directories exist before saving.
CUAD_TRAIN_PATH.mkdir(parents=True, exist_ok=True)
CUAD_VALIDATION_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
save_jsonl(train_data, CUAD_TRAIN_PATH/'cuad_train.jsonl')
save_jsonl(val_data, CUAD_VALIDATION_PATH/'cuad_validation.jsonl')
print(f"Saved {len(train_data)} training samples and {len(val_data)} validation samples.")

# Step 7 & 8 : QLoRA fine-tuning with completion-only loss

Two changes versus the original pipeline:

- **Step 7 — completion-only loss:** the answer is a single token (`Yes`/`No`), so computing loss over the whole prompt lets the gradient be dominated by reproducing the clause text and drowns out the decision signal. We feed the trainer a **prompt/completion** dataset and set `SFTConfig(completion_only_loss=True)`, so trl masks every prompt token (`-100`) and only the answer contributes to the loss. *(In older trl this was done with `DataCollatorForCompletionOnlyLM`, which was **removed in trl 1.x** — `completion_only_loss` is the supported replacement.)*
- **Step 8 — config tidy-ups:** prefer `bf16` over `fp16` for stability, widen LoRA targets to `["q_proj","k_proj","v_proj","o_proj"]`, and lower `max_length` to `1024` (single clauses are short → faster, less memory).

The `### Instruction / ### Input / ### Response` template is unchanged — it now lives inside the `prompt` field, with `### Response:\n` at the boundary where prompt-masking switches off. `load_dataset` points at the same JSONL files written in the save cell above.

> **trl 1.x API note:** `SFTTrainer` now takes an `SFTConfig` (not `TrainingArguments`), the tokenizer is passed as `processing_class=`, and `max_seq_length` moved into the config as `max_length`.

- Note : before execution of the cell below run to the terminal `$env:HF_TOKEN=your_hf_token`

In [ ]:
# Diagnostic: confirm WHICH account the token belongs to and whether it can access the gated repo.
# A 403 "not in the authorized list" means the token is valid but this account lacks access.
import os
from dotenv import load_dotenv
from huggingface_hub import whoami, auth_check
from huggingface_hub.utils import GatedRepoError, HfHubHTTPError

load_dotenv()
hf_token = os.getenv("HF_TOKEN")
assert hf_token, "HF_TOKEN not found — check your .env file"

# Temporary: use a smaller model so the 4-bit weights fit fully in GPU VRAM (no CPU/disk offload). Original: "meta-llama/Meta-Llama-3-8B"
MODEL_ID = "meta-llama/Llama-3.2-1B"

# 1) Which account is this token? Request access on the model page with THIS exact account.
me = whoami(token=hf_token)
print(f"Token belongs to: {me['name']}  (type: {me.get('type')})")

# 2) Does that account actually have access to the gated repo?
try:
    auth_check(MODEL_ID, token=hf_token)
    print(f"✅ Access granted to {MODEL_ID} — you can run the load cell below.")
except GatedRepoError:
    print(f"❌ Still gated for account '{me['name']}'.")
    print(f"   -> Visit https://huggingface.co/{MODEL_ID} while logged in as '{me['name']}', "
          f"accept the license, and wait for approval.")
    print(f"   -> Or use the ungated mirror: model_name = 'unsloth/Llama-3.2-1B'")
except HfHubHTTPError as e:
    print(f"❌ Auth/HTTP error (likely an invalid or expired token): {e}")


In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig   # trl 1.x: SFTConfig replaces TrainingArguments here;
                                        # DataCollatorForCompletionOnlyLM was removed (see 5b/7).
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env from the current working dir (project root)

hf_token = os.getenv("HF_TOKEN")
assert hf_token, "HF_TOKEN not found — check your .env file"

from huggingface_hub import login
login(token=hf_token)

# 1. Configuration
# Temporary: use a smaller model so the 4-bit weights fit fully in GPU VRAM (no CPU/disk offload). Original: "meta-llama/Meta-Llama-3-8B"
model_name = "meta-llama/Llama-3.2-1B" 
new_model_name = "llama-3.2-1B-cuad-task1-smoke-test"
MAX_SEQ_LENGTH = 1024   # single clauses are short; used by the trainer + pre-flight checks

# 2. QLoRA Config (4-bit loading to fit on consumer GPU)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,   # Step 8: bf16 for stability
)

# 3. Load Base Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)
model.config.use_cache = False # Silence warnings during training

# 4. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fix for fp16

# 5. Load Dataset (load the same files that were saved)
dataset = load_dataset("json", data_files={
    "train":      str(CUAD_TRAIN_PATH / "cuad_train.jsonl"),
    "validation": str(CUAD_VALIDATION_PATH / "cuad_validation.jsonl"),
})

# 5b. Convert instruction/input/output -> prompt/completion.
# trl 1.x replaces DataCollatorForCompletionOnlyLM with SFTConfig(completion_only_loss=True):
# when the dataset has `prompt` + `completion` columns, SFTTrainer masks the prompt tokens
# automatically so only the Yes/No answer contributes to the loss. The `### Response:\n`
# marker now sits at the end of `prompt`, exactly where masking switches off.
RESPONSE_TEMPLATE = "### Response:\n"

def to_prompt_completion(ex):
    prompt = (
        f"### Instruction:\n{ex['instruction']}\n\n"
        f"### Input:\n{ex['input']}\n\n"
        f"{RESPONSE_TEMPLATE}"
    )
    return {"prompt": prompt, "completion": ex["output"]}

dataset = dataset.map(
    to_prompt_completion,
    remove_columns=dataset["train"].column_names,
)

# 6. LoRA Configuration
peft_config = LoraConfig(
    r=16,       # Rank (Higher = more parameters to train, 16-64 is standard)
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # Step 8: wider targets for a slightly stronger adapter
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

# 7. SFTConfig (trl 1.x: replaces TrainingArguments AND the completion-only collator)
sft_config = SFTConfig(
    output_dir="./results",
    num_train_epochs=1,           # 1 epoch is often enough for SFT on small datasets
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    weight_decay=0.001,
    bf16=True,                    # Step 8: bf16 over fp16 for stability
    logging_steps=25,
    save_steps=100,
    optim="paged_adamw_32bit",    # Syllabus optimization
    max_length=MAX_SEQ_LENGTH,    # Step 8: was SFTTrainer(max_seq_length=...)
    completion_only_loss=True,    # Step 7: replaces DataCollatorForCompletionOnlyLM
    packing=False,                # keep examples separate so prompt-masking is per-example
)

# 8. Initialize Trainer
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=peft_config,
    processing_class=tokenizer,   # trl 1.x: was tokenizer=...
)

# Step 8b : Pre-flight checks — verify everything is set up before training

A full QLoRA run is slow and a misconfigured collator fails *silently* (it trains on zero unmasked tokens and the loss never moves). This cell asserts every prerequisite up front so failures surface in seconds, not after an hour:

1. **GPU / VRAM** — CUDA is present and has enough memory for an 8B model in 4-bit.
2. **Model** — actually loaded in 4-bit and `use_cache=False`.
3. **Tokenizer** — `pad_token` set, right-padded.
4. **Datasets** — both splits present, non-empty, with `instruction` / `input` / `output` columns and only `Yes`/`No` labels.
5. **Completion-only collator (the critical one)** — the `### Response:\n` template is actually found in tokenized batches and the answer tokens are left **unmasked** (otherwise loss ≡ 0).
6. **Sequence length** — how many examples exceed `MAX_SEQ_LENGTH` and would be truncated.
7. **LoRA** — adapters are attached and the base model is frozen (only a tiny % is trainable).

Run this **before** the train cell. If any assertion fails, fix it before training.

In [ ]:
# def _ok(msg):   print(f"  [OK]   {msg}")
# def _warn(msg): print(f"  [WARN] {msg}")

# print("Running pre-flight checks before training...\n")

# # 1) Hardware / CUDA — QLoRA needs a GPU; 4-bit 8B wants ~6 GB just for weights.
# assert torch.cuda.is_available(), "CUDA not available — QLoRA needs a GPU."
# gpu_name = torch.cuda.get_device_name(0)
# vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
# _ok(f"CUDA available — {gpu_name} ({vram_gb:.1f} GB VRAM)")
# if vram_gb < 12:
#     _warn(f"Only {vram_gb:.1f} GB VRAM — an 8B model in 4-bit may OOM at "
#           f"batch_size={sft_config.per_device_train_batch_size}; lower it or use grad-accum.")

# # 2) Base model — must actually be 4-bit quantized and have caching off for training.
# assert model is not None, "Model not loaded."
# is_4bit = getattr(model, "is_loaded_in_4bit", False) or \
#           any(p.dtype == torch.uint8 for p in model.parameters())
# assert is_4bit, "Model is NOT loaded in 4-bit — check BitsAndBytesConfig / load_in_4bit."
# assert model.config.use_cache is False, "model.config.use_cache must be False during training."
# _ok(f"Base model '{model_name}' loaded in 4-bit, use_cache=False")

# # 3) Tokenizer — a missing pad_token or left padding silently corrupts batched SFT.
# assert tokenizer.pad_token is not None, "Tokenizer has no pad_token."
# assert tokenizer.padding_side == "right", f"padding_side must be 'right', got {tokenizer.padding_side!r}."
# _ok(f"Tokenizer OK — pad_token={tokenizer.pad_token!r}, padding_side='{tokenizer.padding_side}'")

# # 4) Datasets — both splits present, non-empty, prompt/completion schema, only Yes/No labels.
# for split in ("train", "validation"):
#     assert split in dataset, f"Dataset missing '{split}' split."
#     assert len(dataset[split]) > 0, f"'{split}' split is empty."
#     missing = {"prompt", "completion"} - set(dataset[split].column_names)
#     assert not missing, f"'{split}' split missing columns: {missing}"
# labels = set(dataset["train"]["completion"]) | set(dataset["validation"]["completion"])
# assert labels <= {"Yes", "No"}, f"Unexpected labels (Task 1 must be Yes/No): {labels - {'Yes', 'No'}}"
# _ok(f"Datasets OK — train={len(dataset['train'])}, val={len(dataset['validation'])}, labels={labels}")

# # 5) CRITICAL — completion-only loss. With trl 1.x + prompt/completion +
# #    completion_only_loss=True, SFTTrainer masks the PROMPT tokens (-100) and leaves
# #    only the answer tokens contributing to the loss. Pull one real batch from the
# #    trainer's dataloader and confirm BOTH: some tokens unmasked (the answer) AND
# #    some tokens masked (the prompt). If nothing is unmasked the loss is ~0 and the
# #    model learns nothing; if nothing is masked, the prompt is being trained on too.
# batch = next(iter(trainer.get_train_dataloader()))
# labels_t = batch["labels"]
# unmasked = int((labels_t != -100).sum())
# masked   = int((labels_t == -100).sum())
# assert unmasked > 0, ("All labels masked (-100) — loss would be zero. "
#                       "Check completion_only_loss / dataset prompt-completion format.")
# assert masked > 0, ("No labels masked — the prompt is not being masked; "
#                     "completion_only_loss may be off or the data is not prompt-completion.")
# _ok(f"Completion-only loss works — {unmasked} answer token(s) unmasked, "
#     f"{masked} prompt token(s) masked in first batch")

# # 6) Sequence length — examples longer than MAX_SEQ_LENGTH get truncated; if the
# #    answer is at the end, the model trains on a cut-off prompt. Report the count.
# prompts     = dataset["train"]["prompt"]
# completions = dataset["train"]["completion"]
# lengths = [len(tokenizer(p + c)["input_ids"]) for p, c in zip(prompts, completions)]
# over = sum(l > MAX_SEQ_LENGTH for l in lengths)
# _ok(f"Token lengths — max={max(lengths)}, mean={sum(lengths)//len(lengths)}, "
#     f">{MAX_SEQ_LENGTH}: {over}/{len(lengths)} examples")
# if over:
#     _warn(f"{over} train examples exceed MAX_SEQ_LENGTH={MAX_SEQ_LENGTH} and will be truncated.")

# # 7) LoRA — adapters attached and base model frozen (only a tiny % should train).
# trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
# total = sum(p.numel() for p in trainer.model.parameters())
# assert trainable > 0, "No trainable parameters — LoRA adapters not attached."
# assert trainable < 0.05 * total, f"{100*trainable/total:.2f}% trainable — base model not frozen."
# _ok(f"LoRA attached — trainable {trainable:,} / {total:,} ({100*trainable/total:.3f}%)")

# print("\nAll pre-flight checks passed — safe to run the training cell below.")

In [ ]:
# 9. Train and Save  (run only after the pre-flight checks above pass)
print("Starting training...")
train_result = trainer.train()

# Post-training sanity: the loss must actually be a real, finite, non-zero number.
# A training loss that is exactly 0 / NaN means the completion-only masking ate
# every label — exactly the silent failure the pre-flight check guards against.
final_loss = train_result.training_loss
assert final_loss is not None and final_loss == final_loss, f"Training loss is NaN: {final_loss}"
assert final_loss > 0, f"Training loss is {final_loss} — no tokens contributed to the loss."
print(f"Training finished — final training loss: {final_loss:.4f}")

# Save the adapter, then verify the files actually landed on disk.
trainer.model.save_pretrained(new_model_name)
saved = list(Path(new_model_name).glob("adapter_*"))
assert any(p.name == "adapter_model.safetensors" or p.name == "adapter_model.bin" for p in saved), \
    f"No adapter weights found in {new_model_name}/ — save may have failed."
assert (Path(new_model_name) / "adapter_config.json").exists(), \
    f"adapter_config.json missing in {new_model_name}/."
print(f"Model saved to {new_model_name}/ — files: {sorted(p.name for p in Path(new_model_name).iterdir())}")

# Step 6b : Evaluate on validation — per-class precision / recall / F1

Under class imbalance, accuracy and loss are meaningless ("always No" can score 80%+). Generate a `Yes`/`No` prediction for every validation example and report **per-class precision / recall / F1 + a confusion matrix** with `sklearn.metrics.classification_report`.

**How to know the fix worked:** a model that ignores the input should now score ~50%, not ~100%. If validation F1 is near-perfect immediately, re-inspect the inputs — the leak may not be fully closed.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Use cache for faster generation at inference time.
model.config.use_cache = True
model.eval()

def predict(example):
    prompt = (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Input:\n{example['input']}\n\n"
        f"### Response:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=1024).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=3,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    return "Yes" if "yes" in decoded.strip().lower() else "No"

y_true = [ex["output"] for ex in val_data]
y_pred = [predict(ex) for ex in val_data]

print("Per-class precision / recall / F1 on validation:\n")
print(classification_report(y_true, y_pred, labels=["Yes", "No"], zero_division=0))

print("Confusion matrix (rows = true [Yes, No], cols = pred [Yes, No]):")
print(confusion_matrix(y_true, y_pred, labels=["Yes", "No"]))